# Độ lớn mạng và điều chuẩn

**Lab 09 — Học máy ứng dụng**  
**Nguyễn Hoàng Hải — 2474802010099**

Khảo sát kích thước mạng, alpha và các hàm kích hoạt trên dữ liệu hai vầng trăng.

> Chạy các ô theo thứ tự từ thư mục gốc của dự án để các đường dẫn `data/...` được nhận diện. Cài các thư viện `numpy`, `pandas`, `matplotlib` và `scikit-learn` trước khi chạy.

In [ ]:
"""Khao sat do lon mang, dieu chuan alpha va cac activation tren hai vang trang."""
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


def make_mlp(layers, alpha, activation="relu"):
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(hidden_layer_sizes=layers, alpha=alpha,
                      activation=activation, solver="lbfgs",
                      max_iter=20000, random_state=0),
    )


df = pd.read_csv("data/hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print("Ba mang: accuracy train va test")
for name, layers, alpha in [
    ("(2,), alpha=1e-4", (2,), 1e-4),
    ("(100,100), alpha=1e-5", (100, 100), 1e-5),
    ("(100,100), alpha=1", (100, 100), 1.0),
]:
    model = make_mlp(layers, alpha).fit(X_tr, y_tr)
    print(f"{name:24s} {accuracy_score(y_tr, model.predict(X_tr)):.4f} "
          f"{accuracy_score(y_te, model.predict(X_te)):.4f}")

scores = {}
print("\nAlpha kiem dinh cheo:")
for alpha in [1e-5, 1e-3, 0.1, 1.0, 3.0, 10.0]:
    scores[alpha] = cross_val_score(make_mlp((100, 100), alpha), X_tr, y_tr, cv=5).mean()
    print(f"{alpha:<8} {scores[alpha]:.4f}")
best_alpha = max(scores, key=scores.get)
best = make_mlp((100, 100), best_alpha).fit(X_tr, y_tr)
print(f"Alpha tot nhat={best_alpha}; accuracy test={accuracy_score(y_te, best.predict(X_te)):.4f}")

# Bai tap 3: so sanh activation
print("\nBai tap 3: activation | accuracy train | accuracy test")
for activation in ["logistic", "tanh", "relu"]:
    model = make_mlp((100, 100), 1.0, activation).fit(X_tr, y_tr)
    print(f"{activation:10s} | {accuracy_score(y_tr, model.predict(X_tr)):.4f} "
          f"| {accuracy_score(y_te, model.predict(X_te)):.4f}")
